In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

from torch.utils.data import Dataset, TensorDataset, DataLoader, random_split
import time
import json
from sklearn.preprocessing import StandardScaler


In [2]:
path = Path('..//..//results//15062026//nn_inputs')

acc = np.load(Path(path, 'acc_inputs.npy'))

strain = np.load(Path(path, 'strain_inputs.npy'))

temp = np.load(Path(path, 'temperature_inputs.npy'))

event_ids = np.load(Path(path, 'event_ids.npy'))

In [3]:
with open(f'{path}//metadata//sensor_ids.json') as f:
    sensor_names = json.load(f)

In [4]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cpu


In [10]:
#To be honest, the creation of this could also be its own script lol 

class SensorDataset(Dataset):

    def __init__(
        self,
        acc_data,
        strain_data
    ):

        self.acc = torch.tensor(
            acc_data,
            dtype=torch.float32
        )

        self.strain = torch.tensor(
            strain_data,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.acc)

    def __getitem__(self, idx):

        return {
            "acc": self.acc[idx],
            "strain": self.strain[idx]
        }


def get_dataloaders(acc, strain, event_masks, set_idx = 0):

    acc_set = acc[event_masks[set_idx]]
    strain_set = strain[event_masks[set_idx]]

    dataset = SensorDataset(
        acc_set,
        strain_set
    )


    N = len(dataset)

    train_size = int(0.7 * N)
    test_size = int(0.2 * N)
    val_size = N - train_size - test_size


    generator = torch.Generator().manual_seed(
        42
    )

    # Split first
    train_dataset, test_dataset, val_dataset = random_split(
        dataset,
        [train_size, test_size, val_size],
        generator=generator
    )

    # Get indices from splits
    train_idx = train_dataset.indices
    test_idx = test_dataset.indices
    val_idx = val_dataset.indices

    # Extract raw arrays
    acc_train = acc_set[train_idx]
    acc_test = acc_set[test_idx]
    acc_val = acc_set[val_idx]

    strain_train = strain_set[train_idx]
    strain_test = strain_set[test_idx]
    strain_val = strain_set[val_idx]


    acc_scaler = StandardScaler()

    # flatten feature dimension temporarily
    acc_shape = acc_train.shape

    acc_train_scaled = acc_scaler.fit_transform(
        acc_train.reshape(-1, acc_train.shape[-1])
    ).reshape(acc_shape)

    acc_test_scaled = acc_scaler.transform(
        acc_test.reshape(-1, acc_test.shape[-1])
    ).reshape(acc_test.shape)

    acc_val_scaled = acc_scaler.transform(
        acc_val.reshape(-1, acc_val.shape[-1])
    ).reshape(acc_val.shape)


    strain_scaler = StandardScaler()

    strain_shape = strain_train.shape

    strain_train_scaled = strain_scaler.fit_transform(
        strain_train.reshape(-1, strain_train.shape[-1])
    ).reshape(strain_shape)

    strain_test_scaled = strain_scaler.transform(
        strain_test.reshape(-1, strain_test.shape[-1])
    ).reshape(strain_test.shape)

    strain_val_scaled = strain_scaler.transform(
        strain_val.reshape(-1, strain_val.shape[-1])
    ).reshape(strain_val.shape)

    # Rebuild datasets
    train_dataset = SensorDataset(
        acc_train_scaled,
        strain_train_scaled
    )

    test_dataset = SensorDataset(
        acc_test_scaled,
        strain_test_scaled
    )

    val_dataset = SensorDataset(
        acc_val_scaled,
        strain_val_scaled
    )


    train_loader = DataLoader(
        train_dataset,
        batch_size=64,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=64,
        shuffle=False
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=64,
        shuffle=False
    )

    return train_loader, test_loader, val_loader

In [ ]:
event_masks = [
    [event_id.startswith(f"AQUINAS_SET{x}")
     for event_id in event_ids]
    for x in range(1,6)
]



In [11]:
train, test, val = get_dataloaders(acc, strain, event_masks)


In [12]:
train